# Computer Vision Workshop: Building Monkey Thinking

Welcome! In this workshop, we'll build a hand gesture detection app step-by-step. By the end, you'll have written the complete Monkey Thinking application that detects your hand gestures and displays monkey memes in real-time.

## What We're Building

An app that:
- Detects hands using your webcam
- Recognizes the "pointing" gesture
- Shows green dots on your hand landmarks
- Displays monkey memes that change based on your gesture

## Workshop Structure

1. Part 1: Detecting Hand Landmarks
2. Part 2: Working with Images and Colors
3. Part 3: Building the Complete Application

Let's get started!

## Setup: Install Libraries

First, we need to install the required libraries. Run this cell once:

In [ ]:
# Run this cell only once to install packages
# !pip install opencv-python mediapipe numpy

---

# PART 1: Detecting Hand Landmarks

In this section, you'll learn to:
- Set up MediaPipe hand detection
- Capture video from your webcam
- Detect hands and get their landmark positions
- Understand what landmarks are and how to use them

## Step 1.1: Import Libraries

Let's start by importing everything we need:

In [ ]:
import cv2 
import mediapipe as mp 
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("Libraries imported successfully!")

## Step 1.2: Set Up MediaPipe Hand Landmarker

MediaPipe provides a pre-trained model that can detect hands and find 21 landmark points on each hand. Let's configure it:

In [ ]:
# Configure the hand detection model
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options = base_options,
    num_hands = 2,  # Can detect up to 2 hands
    running_mode = vision.RunningMode.VIDEO,
    min_hand_detection_confidence = 0.6,
    min_tracking_confidence = 0.6)

landmarker = vision.HandLandmarker.create_from_options(options)

print("Hand landmarker ready!")

## Step 1.3: Understanding Hand Landmarks

Each hand has 21 landmarks (key points). Here are the important ones we'll use:

```
Landmark Numbers:
- [0] = Wrist
- [4] = Thumb tip
- [8] = Index finger tip
- [12] = Middle finger tip
- [16] = Ring finger tip
- [20] = Pinky tip

Finger joints:
- [6] = Index finger PIP (middle joint)
- [10] = Middle finger PIP
- [14] = Ring finger PIP
- [18] = Pinky PIP
```

For detecting a "pointing" gesture, we check:
- Is index finger extended? (tip is higher than middle joint)
- Are other fingers folded? (tips are lower than middle joints)

In image coordinates, Y=0 is at the top, so:
- `tip.y < joint.y` means finger is extended (pointing up)
- `tip.y > joint.y` means finger is folded

## Step 1.4: Open the Camera

Now let's open your webcam:

In [ ]:
# Open camera (try 0 if 1 doesn't work)
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Error: Could not open camera")
else:
    print("Camera opened successfully!")
    print(f"Camera width: {cap.get(cv2.CAP_PROP_FRAME_WIDTH)}")
    print(f"Camera height: {cap.get(cv2.CAP_PROP_FRAME_HEIGHT)}")

## Step 1.5: Capture and Process One Frame

Let's capture a single frame and detect hands in it:

In [ ]:
# Capture one frame
valid, frame = cap.read()

if valid:
    print("Frame captured!")
    print(f"Frame shape: {frame.shape}")
    
    # Convert BGR to RGB (MediaPipe needs RGB)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Create MediaPipe image
    mp_image = mp.Image(
        image_format = mp.ImageFormat.SRGB,
        data = rgb_frame
    )
    
    # Detect hands
    result = landmarker.detect_for_video(mp_image, 0)
    
    if result.hand_landmarks:
        print(f"Detected {len(result.hand_landmarks)} hand(s)")
        print(f"Each hand has {len(result.hand_landmarks[0])} landmarks")
    else:
        print("No hands detected")

## Step 1.6: Access Individual Landmarks

Let's see how to access specific landmarks:

In [ ]:
if result.hand_landmarks:
    # Get the first hand
    hand = result.hand_landmarks[0]
    
    # Access specific landmarks
    index_tip = hand[8]
    index_pip = hand[6]
    
    print(f"Index finger tip position: x={index_tip.x:.3f}, y={index_tip.y:.3f}")
    print(f"Index finger PIP position: x={index_pip.x:.3f}, y={index_pip.y:.3f}")
    
    # Check if index finger is extended
    if index_tip.y < index_pip.y:
        print("Index finger is EXTENDED (pointing up)!")
    else:
        print("Index finger is FOLDED")

## Step 1.7: Detect Pointing Gesture

Now let's check if the hand is making a pointing gesture:

In [ ]:
if result.hand_landmarks:
    hand = result.hand_landmarks[0]
    
    # Get landmarks for all fingers
    index_tip = hand[8]
    index_pip = hand[6]
    middle_tip = hand[12]
    middle_pip = hand[10]
    ring_tip = hand[16]
    ring_pip = hand[14]
    pinky_tip = hand[20]
    pinky_pip = hand[18]
    
    # Check each finger
    index_extended = index_tip.y < index_pip.y
    middle_folded = middle_tip.y > middle_pip.y
    ring_folded = ring_tip.y > ring_pip.y
    pinky_folded = pinky_tip.y > pinky_pip.y
    
    print(f"Index extended: {index_extended}")
    print(f"Middle folded: {middle_folded}")
    print(f"Ring folded: {ring_folded}")
    print(f"Pinky folded: {pinky_folded}")
    
    # Pointing = only index finger extended
    if index_extended and middle_folded and ring_folded and pinky_folded:
        print("\n*** POINTING GESTURE DETECTED! ***")
    else:
        print("\nNot pointing")

---

# PART 2: Working with Images and Colors

In this section, you'll learn to:
- Load images with OpenCV
- Resize images
- Draw on images (circles, text, etc.)
- Combine multiple images side-by-side
- Display images in a window

## Step 2.1: Load a Meme Image

Let's load one of our monkey meme images:

In [ ]:
# Load an image
meme_image = cv2.imread("meme/staring.png")

if meme_image is None:
    print("Error: Could not load meme image")
else:
    print("Meme loaded successfully!")
    print(f"Meme image shape: {meme_image.shape}")
    print(f"Height: {meme_image.shape[0]}, Width: {meme_image.shape[1]}")

## Step 2.2: Display an Image

Let's show the image in a window:

In [ ]:
# Display the image
cv2.imshow('Meme', meme_image)
cv2.waitKey(0)  # Wait for any key press
cv2.destroyAllWindows()

print("Image displayed! (Close the window to continue)")

## Step 2.3: Resize an Image

Images need to be the same size to combine them. Let's resize:

In [ ]:
# Set target size
target_width = 640
target_height = 480

# Resize the meme
meme_resized = cv2.resize(meme_image, (target_width, target_height))

print(f"Original size: {meme_image.shape}")
print(f"Resized to: {meme_resized.shape}")

## Step 2.4: Draw Circles on an Image

Let's practice drawing on our camera frame. We'll draw green circles:

In [ ]:
# Make a copy of the frame so we don't modify the original
frame_copy = frame.copy()

# Draw some circles
# cv2.circle(image, (x, y), radius, (B, G, R), thickness)
cv2.circle(frame_copy, (100, 100), 5, (0, 255, 0), -1)  # Green circle at (100, 100)
cv2.circle(frame_copy, (200, 150), 5, (0, 255, 0), -1)  # Green circle at (200, 150)
cv2.circle(frame_copy, (300, 200), 5, (0, 255, 0), -1)  # Green circle at (300, 200)

# Display
cv2.imshow('Frame with Circles', frame_copy)
cv2.waitKey(0)
cv2.destroyAllWindows()

print("Circles drawn! Note: color is (B, G, R) not (R, G, B)")

In [ ]:
if result.hand_landmarks:
    frame_with_landmarks = frame.copy()
    
    for hand in result.hand_landmarks:
        # Loop through all 21 landmarks
        for lm in hand:
            # Convert normalized coordinates to pixel coordinates
            h, w, _ = frame.shape
            cx = int(lm.x * w)
            cy = int(lm.y * h)
            
            # Draw green circle at landmark position
            cv2.circle(frame_with_landmarks, (cx, cy), 5, (0, 255, 0), -1)
    
    cv2.imshow('Hand Landmarks', frame_with_landmarks)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    print("Drew green circles on all hand landmarks!")

## Step 2.6: Combine Two Images Side-by-Side

Let's combine the meme and camera frame horizontally:

In [ ]:
# First, make sure both images are the same height
frame_height, frame_width = frame.shape[:2]
meme_resized = cv2.resize(meme_image, (frame_width, frame_height))

# Stack them horizontally
combined = np.hstack([meme_resized, frame])

print(f"Meme size: {meme_resized.shape}")
print(f"Frame size: {frame.shape}")
print(f"Combined size: {combined.shape}")

# Display
cv2.imshow('Side by Side', combined)
cv2.waitKey(0)
cv2.destroyAllWindows()

print("Images combined side-by-side!")

---

# PART 3: Building the Complete Application

Now we'll put everything together to create the full Monkey Thinking app!

The app will:
1. Continuously capture frames from the webcam
2. Detect hands in each frame
3. Check for pointing gesture
4. Load the appropriate meme (pointing or staring)
5. Draw green circles on hand landmarks
6. Display meme and camera feed side-by-side
7. Run until you press 'q'

## Step 3.1: The Complete Code - Explained

Here's the full application. Read through each section:

In [ ]:
# ===== IMPORTS =====
import cv2 
import mediapipe as mp 
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ===== LOAD INITIAL MEME =====
meme_image = cv2.imread("meme/staring.png")
if meme_image is None:
    print("Error: Could not load meme image")
    exit()

# ===== SET UP HAND DETECTION =====
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')

options = vision.HandLandmarkerOptions(
    base_options = base_options,
    num_hands = 2,
    running_mode = vision.RunningMode.VIDEO,
    min_hand_detection_confidence = 0.6,
    min_tracking_confidence = 0.6)

landmarker = vision.HandLandmarker.create_from_options(options)

# ===== OPEN CAMERA =====
cap = cv2.VideoCapture(1)  # Try 0 if 1 doesn't work
timestamp = 0

if not cap.isOpened():
    print("Error: Could not open camera")
    exit()

print("Camera opened successfully!")
print(f"Camera width: {cap.get(cv2.CAP_PROP_FRAME_WIDTH)}")
print(f"Camera height: {cap.get(cv2.CAP_PROP_FRAME_HEIGHT)}")
print("\nPress 'q' to quit")

# ===== MAIN LOOP =====
while cap.isOpened():
    # Capture frame
    valid, frame = cap.read()
    
    if not valid:
        print("Warning: Could not read frame")
        break
    
    # Convert to RGB for MediaPipe
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(
        image_format = mp.ImageFormat.SRGB,
        data = rgb_frame
    )
    
    # Detect hands
    result = landmarker.detect_for_video(mp_image, timestamp)
    timestamp += 1
    
    # Process detected hands
    if result.hand_landmarks:
        for hand in result.hand_landmarks:
            # Get finger landmarks
            index_tip = hand[8]
            index_pip = hand[6]
            middle_tip = hand[12]
            middle_pip = hand[10]
            ring_tip = hand[16]
            ring_pip = hand[14]
            pinky_tip = hand[20]
            pinky_pip = hand[18]
            
            # Check finger positions
            index_extended = index_tip.y < index_pip.y
            middle_folded = middle_tip.y > middle_pip.y
            ring_folded = ring_tip.y > ring_pip.y
            pinky_folded = pinky_tip.y > pinky_pip.y
            
            # Change meme based on gesture
            if index_extended and middle_folded and ring_folded and pinky_folded:
                meme_image = cv2.imread("meme/pointing.png")
            else:
                meme_image = cv2.imread("meme/staring.png")
            
            # Draw landmarks
            for lm in hand:
                h, w, _ = frame.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
    else:
        # No hands detected - show staring meme
        meme_image = cv2.imread("meme/staring.png")
    
    # Combine meme and frame
    frame_height, frame_width = frame.shape[:2]
    meme_resized = cv2.resize(meme_image, (frame_width, frame_height))
    combined = np.hstack([meme_resized, frame])
    
    # Display
    cv2.imshow('Think Monke', combined)
    
    # Check for quit key
    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

# ===== CLEANUP =====
cap.release()
cv2.destroyAllWindows()
landmarker.close()
print("Application closed!")

## Step 3.2: Code Walkthrough

Let's break down what happens in each part:

### Initialization (Lines 1-29)
- Import libraries
- Load the starting meme image
- Set up the hand detector
- Open the camera

### Main Loop (Lines 32-88)
This runs continuously until you press 'q':

1. **Capture Frame**: Get a new image from the camera
2. **Convert Color**: Change BGR to RGB for MediaPipe
3. **Detect Hands**: Find hands and landmark positions
4. **Check Gesture**: If hands detected, check if pointing
5. **Load Meme**: Load pointing.png or staring.png based on gesture
6. **Draw Landmarks**: Draw green circles on all landmarks
7. **Combine Images**: Put meme and camera side-by-side
8. **Display**: Show the result
9. **Check Quit**: If 'q' pressed, exit loop

### Cleanup (Lines 91-94)
- Release the camera
- Close all windows
- Close the hand detector

## Step 3.3: Run Your Application

The complete code is above in Step 3.1. To run it:

1. Make sure you have the required files:
   - `hand_landmarker.task` (model file)
   - `meme/staring.png` (default meme)
   - `meme/pointing.png` (pointing gesture meme)

2. Run the cell in Step 3.1

3. Try making a pointing gesture with your index finger

4. Press 'q' to quit

Congratulations! You've built a complete computer vision application!

## Challenge: Extend the App

Now that you've built the app, try these challenges:

### Challenge 1: Add More Memes
Add detection for a peace sign (index + middle finger extended) and load `meme/thinking.png`

### Challenge 2: Change Circle Color
Make the landmark circles a different color (try blue: (255, 0, 0))

### Challenge 3: Add Text Display
Use `cv2.putText()` to display "POINTING!" on the screen when the gesture is detected

### Challenge 4: Try Different Gestures
Experiment with detecting thumbs up or an open palm

Good luck and have fun!

## Summary

In this workshop, you learned:

### Part 1: Hand Landmark Detection
- Set up MediaPipe for hand detection
- Understand the 21 hand landmarks
- Access landmark coordinates
- Detect specific gestures by comparing landmark positions

### Part 2: Images and Colors
- Load and display images with OpenCV
- Resize images to match sizes
- Draw shapes (circles) on images
- Combine images side-by-side

### Part 3: Complete Application
- Build a real-time video processing loop
- Integrate hand detection with visual feedback
- Create an interactive gesture-controlled app

You now have a working computer vision application! Keep experimenting and building more features.

In [ ]:
# Make a copy of the frame so we don't modify the original
frame_copy = frame.copy()

# Draw some circles
# cv2.circle(image, (x, y), radius, (B, G, R), thickness)
cv2.circle(frame_copy, (100, 100), 5, (0, 255, 0), -1)  # Green circle at (100, 100)
cv2.circle(frame_copy, (200, 150), 5, (0, 255, 0), -1)  # Green circle at (200, 150)
cv2.circle(frame_copy, (300, 200), 5, (0, 255, 0), -1)  # Green circle at (300, 200)

# Display
cv2.imshow('Frame with Circles', frame_copy)
cv2.waitKey(0)
cv2.destroyAllWindows()

print("Circles drawn! Note: color is (B, G, R) not (R, G, B)")